In [1]:
# python ./gen_pattern.py -a A4 -c 5 -r 7 -T charuco_board -u mm -s 40 -p 25 -f DICT_6X6_50.json.gz

import cv2
import numpy as np
from pathlib import Path

In [20]:
# board = cv2.aruco.CharucoBoard((5, 7), 40 / 1000.0, 25 / 1000.0, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_50))
board = cv2.aruco.CharucoBoard((5, 7), 40 / 1000.0 * 0.9, 25 / 1000.0 * 0.9, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_50))
# charuco_params = cv2.aruco.CharucoParameters()
# detector_params = cv2.aruco.DetectorParameters()
detector = cv2.aruco.CharucoDetector(board)

all_charuco_corners = []
all_charuco_ids = []
all_image_points = []
all_object_points = []
all_image_sizes = []

images = sorted(Path('./biaoding/').glob('*.png'))
for image in images[:1]:
    img = cv2.imread(str(image))

    charuco_corners, charuco_ids, marker_corners, marker_ids = detector.detectBoard(img)
    assert charuco_corners is not None, "No charuco corners detected"
    object_points, image_points = board.matchImagePoints(charuco_corners, charuco_ids)
    assert object_points is not None, "No object points detected"

    all_charuco_corners.append(charuco_corners)
    all_charuco_ids.append(charuco_ids)
    all_image_points.append(image_points)
    all_object_points.append(object_points)
    all_image_sizes.append(img.shape[:2])

all_image_sizes = set(all_image_sizes)
assert len(all_image_sizes) == 1, "All images must have the same size"
image_size = all_image_sizes.pop()

In [28]:
object_points

array([[[0.036     , 0.036     , 0.        ]],

       [[0.072     , 0.036     , 0.        ]],

       [[0.108     , 0.036     , 0.        ]],

       [[0.144     , 0.036     , 0.        ]],

       [[0.036     , 0.072     , 0.        ]],

       [[0.072     , 0.072     , 0.        ]],

       [[0.108     , 0.072     , 0.        ]],

       [[0.144     , 0.072     , 0.        ]],

       [[0.036     , 0.108     , 0.        ]],

       [[0.072     , 0.108     , 0.        ]],

       [[0.108     , 0.108     , 0.        ]],

       [[0.144     , 0.108     , 0.        ]],

       [[0.036     , 0.144     , 0.        ]],

       [[0.072     , 0.144     , 0.        ]],

       [[0.108     , 0.144     , 0.        ]],

       [[0.144     , 0.144     , 0.        ]],

       [[0.036     , 0.17999999, 0.        ]],

       [[0.072     , 0.17999999, 0.        ]],

       [[0.108     , 0.17999999, 0.        ]],

       [[0.144     , 0.17999999, 0.        ]],

       [[0.036     , 0.21599999, 0.     

In [3]:
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(all_object_points, all_image_points, image_size, None, None, flags=cv2.CALIB_FIX_K1+cv2.CALIB_FIX_K2+cv2.CALIB_FIX_K3+cv2.CALIB_ZERO_TANGENT_DIST)

In [4]:
ret, mtx, dist, rvecs, tvecs

(0.34313423105984175,
 array([[858.50950806,   0.        , 635.07246546],
        [  0.        , 855.73723496, 372.97716583],
        [  0.        ,   0.        ,   1.        ]]),
 array([[0., 0., 0., 0., 0.]]),
 (array([[-0.03412414],
         [ 0.16768742],
         [ 3.01901814]]),
  array([[-0.06149962],
         [-0.01188473],
         [-2.70542793]]),
  array([[ 0.12278159],
         [-0.16773715],
         [-3.05776587]]),
  array([[ 0.45666022],
         [-0.22833776],
         [-1.00689139]]),
  array([[ 0.42364851],
         [-0.13596717],
         [-0.76638926]]),
  array([[ 0.42772408],
         [-0.09734772],
         [-0.59683783]]),
  array([[ 0.43517728],
         [-0.05638158],
         [-0.40878088]]),
  array([[ 0.43802569],
         [-0.01290617],
         [-0.2159816 ]]),
  array([[0.43447058],
         [0.05481521],
         [0.07631137]]),
  array([[0.44466856],
         [0.03424816],
         [0.37462886]]),
  array([[0.53445205],
         [0.02453557],
        

In [7]:
mtx = np.array([[906.55889893,   0.        , 653.27850342],
                [  0.        , 906.25061035, 366.25604248],
                [  0.        ,   0.        ,   1.        ]])

In [8]:
img = cv2.imread("Data Collection_screenshot_15.10.2025-2.png")
charuco_corners, charuco_ids, marker_corners, marker_ids = detector.detectBoard(img)
assert charuco_corners is not None, "No charuco corners detected"
# now use solvePnP to get the pose
object_points, image_points = board.matchImagePoints(charuco_corners, charuco_ids)
assert object_points is not None, "No object points detected"
np.sqrt(cv2.solvePnP(object_points, image_points, mtx, dist)[2][:, 0] @ cv2.solvePnP(object_points, image_points, mtx, dist)[2][:, 0]
)
# cv2.imwrite(f"Data Collection_screenshot_15.10.2025-2.png", 

np.float64(0.41480778911603006)

In [ ]:
charuco_ids

20

In [15]:
_, rvec, tvec = cv2.solvePnP(object_points, image_points, mtx, dist)
RT = np.eye(4)
R, _ = cv2.Rodrigues(rvec)
T = tvec[:, 0]
RT[:3, :3] = R
RT[:3, 3] = T
RT

array([[ 0.20583422, -0.92975847,  0.3052564 ,  0.15516462],
       [ 0.92076131,  0.07836774, -0.38217419, -0.17597855],
       [ 0.33140744,  0.3597328 ,  0.87221638,  0.34208332],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [16]:
-R.T @ T

array([ 0.0167271 ,  0.03499807, -0.41299012])

In [14]:
np.linalg.inv(RT)

array([[ 0.20583422,  0.92076131,  0.33140744,  0.0167271 ],
       [-0.92975847,  0.07836774,  0.3597328 ,  0.03499807],
       [ 0.3052564 , -0.38217419,  0.87221638, -0.41299012],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [7]:
mtx

array([[858.51607084,   0.        , 635.07438454],
       [  0.        , 855.74345864, 372.97561717],
       [  0.        ,   0.        ,   1.        ]])

In [8]:
ret

0.3431607718568679

In [9]:
# img = cv2.imread("biaoding/Data Collection_screenshot_15.10.2025.png")
# charuco_corners, charuco_ids, marker_corners, marker_ids = detector.detectBoard(img)
# assert charuco_corners is not None, "No charuco corners detected"
# object_points, image_points = board.matchImagePoints(charuco_corners, charuco_ids)
# assert object_points is not None, "No object points detected"
# # cv2.solvePnP(object_points, image_points, mtx, dist)
# cv2.imwrite(f"detect/Data Collection_screenshot_15.10.2025.png", cv2.aruco.drawDetectedCornersCharuco(img, charuco_corners, charuco_ids))

In [10]:
img.shape

(720, 1280, 3)

In [11]:
mtx

array([[858.51607084,   0.        , 635.07438454],
       [  0.        , 855.74345864, 372.97561717],
       [  0.        ,   0.        ,   1.        ]])

In [12]:
# visualize
for image, charuco_corners, charuco_ids in zip(images, all_charuco_corners, all_charuco_ids):
    img = cv2.imread(str(image))
    cv2.imwrite(f"detect/{image.name}", cv2.aruco.drawDetectedCornersCharuco(img, charuco_corners, charuco_ids))
    cv2.imwrite(f"undistort/{image.name}", cv2.undistort(img, mtx, dist))

In [13]:
mtx

array([[858.51607084,   0.        , 635.07438454],
       [  0.        , 855.74345864, 372.97561717],
       [  0.        ,   0.        ,   1.        ]])

In [13]:
rvecs

(array([[-0.39914813],
        [-0.0324101 ],
        [-0.26399698]]),
 array([[-0.35443586],
        [ 0.10007694],
        [ 0.46044497]]),
 array([[-0.20332305],
        [-0.08301474],
        [-0.48617761]]),
 array([[-0.21608516],
        [ 0.10657995],
        [ 0.7202085 ]]),
 array([[-0.33380116],
        [-0.13939359],
        [-1.13835783]]),
 array([[-0.57611964],
        [-0.07165952],
        [-0.44893308]]),
 array([[-0.51167904],
        [ 0.1489844 ],
        [ 0.47484068]]),
 array([[-0.29919014],
        [-0.04610668],
        [ 0.01911451]]),
 array([[ 0.16952563],
        [-0.06306451],
        [-0.15015491]]),
 array([[0.07778577],
        [0.30559192],
        [0.05473917]]))

In [18]:
marker_ids

array([[ 4],
       [ 9],
       [ 1],
       [14],
       [ 6],
       [11],
       [ 3],
       [16],
       [ 0],
       [ 8],
       [13],
       [ 5],
       [10],
       [ 2],
       [15],
       [ 7],
       [12]], dtype=int32)

In [ ]:
detector_params.